# Getting started: security posture of .lt websites

This notebook loads the open dataset from a passive security survey of websites under the Lithuanian `.lt` domain and reproduces its headline numbers: what the sites are built with, how many leave internal files readable, and how many domains publish SPF and DMARC records.

The dataset is anonymised. It has one row per hostname and no domain names, IP addresses or URLs, so a row cannot be traced to a particular website.

- Tables for every question, with method notes: https://aguard.lt/en/data
- Ideas for coursework and theses: https://aguard.lt/en/data/research-topics
- Archived copy with a DOI: https://doi.org/10.5281/zenodo.24282573

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt

path = glob.glob('/kaggle/input/**/sites.csv', recursive=True)[0]
df = pd.read_csv(path, dtype={'wp_branch': str, 'php_branch': str})
print(df.shape)
df.head()

## Which hosts were inspected

`status` says how far the check got. Findings exist only for hosts whose home page was inspected, so most of the analysis below uses that subset.

In [ ]:
print(df['status'].value_counts())
ins = df[df['severity'].notna()].copy()
print('home page inspected:', len(ins))

## Overall posture

`severity` is the highest level among a host's findings: 3 is a critical exposure (a readable log, a `.git` directory, a downloadable backup), 2 is outdated software or a disclosed admin surface, 1 is a configuration gap, 0 is none of the listed findings.

In [ ]:
sev = ins['severity'].astype(int).value_counts().sort_index(ascending=False)
print((sev / len(ins) * 100).round(1))
sev.plot.barh(title='Hosts by highest severity'); plt.xlabel('hosts'); plt.show()

## Platforms, and how posture differs between them

Platforms recognised on fewer than 300 hosts are grouped as `other`. `not_detected` means none of the signatures matched, not that the site has no CMS.

In [ ]:
by_cms = ins.groupby('cms').agg(hosts=('id', 'size'),
                                 critical_pct=('severity', lambda s: (s == 3).mean() * 100),
                                 medium_or_critical_pct=('severity', lambda s: (s >= 2).mean() * 100))
by_cms.sort_values('hosts', ascending=False).round(1)

## WordPress branches

Only sites that disclose a version are counted. A site on an older branch is not necessarily vulnerable, but it is not receiving the current release.

In [ ]:
wp = ins[(ins['cms'] == 'WordPress') & ins['wp_branch'].notna()]
print('WordPress sites disclosing a version:', len(wp))
(wp['wp_branch'].value_counts().head(12) / len(wp) * 100).round(1)

## Exposed internal files

Each finding column is 1 or 0. An exposure was recorded only when the response body matched the expected content, not on the status code alone.

In [ ]:
cols = [c for c in ins.columns if c.endswith('_exposed')]
exposed = ins[cols].sum().astype(int).sort_values(ascending=False)
print(exposed)
exposed.sort_values().plot.barh(title='Hosts with an exposed file'); plt.xlabel('hosts'); plt.show()

## SPF and DMARC

Mail records are checked for registered domains only (`host_type == 'domain'`), not for subdomains. A `none` policy only monitors; `quarantine` and `reject` actually stop forged mail.

In [ ]:
mail = df[(df['host_type'] == 'domain') & df['dmarc_policy'].notna()]
print('domains with mail records checked:', len(mail))
print((mail['dmarc_policy'].value_counts() / len(mail) * 100).round(1))
print((mail['spf'].value_counts() / len(mail) * 100).round(1))

## Does the hosting network matter?

Networks announcing fewer than 1,000 hosts are grouped as `other`. A CDN such as Cloudflare appears here in place of the origin host, so this compares what answers the request, not always where the site lives.

In [ ]:
by_net = ins.groupby('network').agg(hosts=('id', 'size'),
                                     critical_pct=('severity', lambda s: (s == 3).mean() * 100),
                                     medium_or_critical_pct=('severity', lambda s: (s >= 2).mean() * 100))
by_net.sort_values('hosts', ascending=False).round(1)

## Limits to keep in mind

- The host list comes from Common Crawl, not from the registry zone file, so it leans towards sites that are linked from elsewhere.
- This is one snapshot (14 to 16 November 2026). It shows a state, not a trend.
- Version columns hold only what a site discloses. Sites that hide versions are not counted as up to date or outdated.

The full method and the data dictionary are in the dataset description and at https://aguard.lt/en/data. If you use the data, please cite: AGuard.lt (2026). Security posture of .lt websites, snapshot 2026-11-16. Zenodo. https://doi.org/10.5281/zenodo.24282573